# Neural Network per K-Means Cluster

**Prerequisite:** Run `kmean.ipynb` first — it saves the processed feature matrix, log target, and cluster labels to `cluster_outputs/`.

This notebook loads those artifacts directly and trains a **separate 5-layer neural network** for each cluster.

| Cluster | Learning Rate | Dropout |
|---------|--------------|--------|
| 0 | 1e-3 | 0.2 |
| 1 | 5e-4 | 0.3 |

In [ ]:
import sys, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S',
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

print('Libraries imported.')

## 1. Load Artifacts from kmean.ipynb

In [ ]:
import os
ARTIFACTS = 'cluster_outputs'

required = ['X_features.npy', 'y_log.npy', 'cluster_labels.npy', 'feature_cols.json']
missing  = [f for f in required if not os.path.exists(f'{ARTIFACTS}/{f}')]
if missing:
    raise FileNotFoundError(
        f'Missing artifacts: {missing}\n'
        'Please run kmean.ipynb first to generate them.'
    )

X_full         = np.load(f'{ARTIFACTS}/X_features.npy')
y_log          = np.load(f'{ARTIFACTS}/y_log.npy')          # log1p(copiesSold)
cluster_labels = np.load(f'{ARTIFACTS}/cluster_labels.npy')

with open(f'{ARTIFACTS}/feature_cols.json') as f:
    feature_cols = json.load(f)

N_CLUSTERS = len(np.unique(cluster_labels))

print(f'X_full         : {X_full.shape}')
print(f'y_log          : {y_log.shape}  (mean={y_log.mean():.3f})')
print(f'cluster_labels : {cluster_labels.shape}')
print(f'N_CLUSTERS     : {N_CLUSTERS}')
print(f'feature_cols   : {len(feature_cols)} features')
print()
for i in range(N_CLUSTERS):
    n = (cluster_labels == i).sum()
    print(f'  Cluster {i}: {n:>7,} games  ({100*n/len(cluster_labels):.1f}%)')

## 2. Neural Network Configurations per Cluster

Both clusters use a **5-layer MLP** (`512 → 256 → 128 → 64 → 32 → 1`) — different learning rate and dropout:

In [ ]:
from src.models.nn import NeuralNetModel

CLUSTER_NN_CONFIGS = {
    0: dict(
        name='Cluster0_NN',
        hidden_layers=[512, 256, 128, 64, 32],
        lr=1e-3,
        dropout_rate=0.2,
        batch_norm=True,
        batch_size=256,
        max_epochs=150,
        patience=15,
    ),
    1: dict(
        name='Cluster1_NN',
        hidden_layers=[512, 256, 128, 64, 32],
        lr=5e-4,
        dropout_rate=0.3,
        batch_norm=True,
        batch_size=256,
        max_epochs=150,
        patience=15,
    ),
}

print(f'  {"Cluster":<10} {"LR":<10} {"Dropout":<10} {"Architecture"}')
print('-' * 65)
for c, cfg in CLUSTER_NN_CONFIGS.items():
    arch = ' → '.join(str(h) for h in cfg['hidden_layers']) + ' → 1'
    print(f'  {c:<10} {cfg["lr"]:<10} {cfg["dropout_rate"]:<10} {arch}')

## 3. Train One Neural Network per Cluster

In [ ]:
trained_models = {}
cluster_data_splits = {}
results = []

for cluster_id in range(N_CLUSTERS):
    print(f'\n{"="*60}')
    print(f'CLUSTER {cluster_id}')
    print(f'{"="*60}')

    mask = cluster_labels == cluster_id
    X_c  = X_full[mask]
    y_c  = y_log[mask]   # log1p(copiesSold)
    print(f'Samples : {len(X_c):,}')

    # 70 / 15 / 15 split within each cluster
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X_c, y_c, test_size=0.30, random_state=42)
    X_val, X_tst, y_val, y_tst = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=42)

    cluster_data_splits[cluster_id] = {
        'X_train': X_tr, 'y_train': y_tr,
        'X_val':   X_val, 'y_val':   y_val,
        'X_test':  X_tst, 'y_test':  y_tst,
    }
    print(f'Train={len(X_tr):,}  Val={len(X_val):,}  Test={len(X_tst):,}')

    cfg   = CLUSTER_NN_CONFIGS[cluster_id]
    model = NeuralNetModel(**cfg)
    print(f'lr={cfg["lr"]}  dropout={cfg["dropout_rate"]}')

    model.fit(X_tr, y_tr, X_val=X_val, y_val=y_val, verbose=True)

    val_metrics  = model.evaluate(X_val, y_val, split_name='val')
    test_metrics = model.evaluate(X_tst, y_tst, split_name='test')

    trained_models[cluster_id] = model
    results.append({
        'Cluster':      cluster_id,
        'N_train':      len(X_tr),
        'LR':           cfg['lr'],
        'Dropout':      cfg['dropout_rate'],
        'Val_RMSE_log': round(val_metrics['rmse_log'],  4),
        'Val_MAE_log':  round(val_metrics['mae_log'],   4),
        'Val_R2':       round(val_metrics['r2_log'],    4),
        'Val_RMSE_raw': round(val_metrics['rmse_raw'],  0),
        'Val_MAE_raw':  round(val_metrics['mae_raw'],   0),
        'Test_RMSE_log':round(test_metrics['rmse_log'], 4),
        'Test_R2':      round(test_metrics['r2_log'],   4),
        'Test_MAE_raw': round(test_metrics['mae_raw'],  0),
    })

print('\n✓ Both cluster models trained.')

## 4. Results Comparison Table

In [ ]:
results_df = pd.DataFrame(results).set_index('Cluster')
print('Per-Cluster Neural Network Performance:')
print('=' * 65)
display(results_df.style
    .background_gradient(cmap='RdYlGn_r', subset=['Val_RMSE_log', 'Test_RMSE_log'])
    .background_gradient(cmap='RdYlGn',   subset=['Val_R2', 'Test_R2'])
    .format({
        'Val_RMSE_log':  '{:.4f}',
        'Val_MAE_log':   '{:.4f}',
        'Val_R2':        '{:.4f}',
        'Val_RMSE_raw':  '{:,.0f}',
        'Val_MAE_raw':   '{:,.0f}',
        'Test_RMSE_log': '{:.4f}',
        'Test_R2':       '{:.4f}',
        'Test_MAE_raw':  '{:,.0f}',
        'LR':            '{:.0e}',
    })
)

## 5. Training Loss Curves per Cluster

In [ ]:
colors = ['#2196F3', '#4CAF50']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for cluster_id, model in trained_models.items():
    ax  = axes[cluster_id]
    cfg = CLUSTER_NN_CONFIGS[cluster_id]
    ax.plot(model.history.get('train_loss', []),
            color=colors[cluster_id], linewidth=2, label='Train')
    if model.history.get('val_loss'):
        ax.plot(model.history['val_loss'], color=colors[cluster_id],
                linewidth=2, linestyle='--', alpha=0.7, label='Val')
    ax.set_title(f'Cluster {cluster_id}\nlr={cfg["lr"]}  dropout={cfg["dropout_rate"]}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Training & Validation Loss — Per-Cluster NNs',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Predicted vs Actual per Cluster

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for cluster_id, model in trained_models.items():
    ax     = axes[cluster_id]
    splits = cluster_data_splits[cluster_id]
    y_true = splits['y_test']
    y_pred = model.predict(splits['X_test'])

    ax.scatter(y_true, y_pred, alpha=0.2, s=6, color=colors[cluster_id])
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect')
    ax.set_xlabel('Actual log(copiesSold+1)', fontsize=10)
    ax.set_ylabel('Predicted log(copiesSold+1)', fontsize=10)

    r2  = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    ax.set_title(f'Cluster {cluster_id}  R²={r2:.3f}  MAE={mae:.3f}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Predicted vs Actual — Per-Cluster NNs (Test Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Export Predictions to CSV

In [ ]:
import os
os.makedirs('cluster_outputs', exist_ok=True)

all_preds = []
for cluster_id, model in trained_models.items():
    mask   = cluster_labels == cluster_id
    X_c    = X_full[mask]
    y_c    = y_log[mask]
    y_pred = model.predict(X_c)

    pred_df = pd.DataFrame({
        'cluster':             cluster_id,
        'actual_log':          y_c,
        'predicted_log':       y_pred,
        'actual_copies':       np.expm1(y_c),
        'predicted_copies':    np.expm1(y_pred),
        'residual_log':        y_pred - y_c,
    })
    path = f'cluster_outputs/cluster_{cluster_id}_nn_predictions.csv'
    pred_df.to_csv(path, index=False)
    all_preds.append(pred_df)
    print(f'Cluster {cluster_id} → {path}  ({len(pred_df):,} rows)')

combined = pd.concat(all_preds, ignore_index=True)
combined.to_csv('cluster_outputs/all_clusters_nn_predictions.csv', index=False)
print(f'\nCombined → cluster_outputs/all_clusters_nn_predictions.csv  ({len(combined):,} rows)')